In [7]:
FILTER_ZEROS = False
THRESHOLD = 1

In [8]:
import pandas as pd
import glob
import numpy as np

files = glob.glob("./unconstrained_results/*.parquet")

df_all = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(df_all.shape)
df_all.head()

(328770, 26)


,Date,Origin,Destination,Scheduled_Flights,Final_True_Demand,Final_Constrained_Bookings,Is_Censored,BOH_DP15,BOH_DP14,BOH_DP13,...,BOH_DP6,BOH_DP5,BOH_DP4,BOH_DP3,BOH_DP2,BOH_DP1,Fuzzy_Capacity,Naive_Est,EM_Est,MARSS_Est
0,2016-01-01,CAN,CGO,1,17676.0,17676.0,False,2644.0,4114.0,5464.0,...,14036.0,15026.0,15913.0,16677.0,17290.0,17676.0,100000.0,NaN,NaN,NaN
1,2016-01-02,CAN,CGO,1,11673.0,11673.0,False,1746.0,2717.0,3608.0,...,9269.0,9922.0,10508.0,11013.0,11418.0,11673.0,100000.0,NaN,NaN,NaN
2,2016-01-03,CAN,CGO,1,7831.0,7831.0,False,1171.0,1822.0,2421.0,...,6218.0,6656.0,7049.0,7388.0,7660.0,7831.0,100000.0,NaN,NaN,NaN
3,2016-01-04,CAN,CGO,1,14482.0,14482.0,False,2166.0,3370.0,4477.0,...,11500.0,12310.0,13037.0,13663.0,14166.0,14482.0,100000.0,NaN,NaN,NaN
4,2016-01-05,CAN,CGO,1,16017.0,16017.0,False,2396.0,3728.0,4951.0,...,12719.0,13615.0,14419.0,15112.0,15667.0,16017.0,100000.0,NaN,NaN,NaN


In [9]:
models = ["Naive_Est", "EM_Est", "MARSS_Est"]

results = []

# Evaluate route-by-route first
for col in models:

    route_metrics = []

    for (o, d), g in df_all.groupby(["Origin", "Destination"]):

        y_true = g["Final_True_Demand"]
        y_pred = g[col]

        mask = y_pred.notna()

        if mask.sum() == 0:
            continue

        yt = y_true[mask].values
        yp = y_pred[mask].values

        mse = np.mean((yt - yp) ** 2)
        mae = np.mean(np.abs(yt - yp))
        rmse = np.sqrt(mse)

        # Optional filter:
        # remove OD pairs that are essentially zero-error / trivial
        if FILTER_ZEROS and (mae + mse + rmse < THRESHOLD):
            print(f"Skipping OD pair ({o} -> {d}) due to low error: MAE={mae:.4f}, MSE={mse:.4f}, RMSE={rmse:.4f}")
            continue

        route_metrics.append((mae, mse, rmse))

    # Aggregate remaining OD pairs
    if len(route_metrics) == 0:
        avg_mae = np.nan
        avg_mse = np.nan
        avg_rmse = np.nan
        n_routes = 0
    else:
        arr = np.array(route_metrics)
        avg_mae = arr[:, 0].mean()
        avg_mse = arr[:, 1].mean()
        avg_rmse = arr[:, 2].mean()
        n_routes = len(route_metrics)

    results.append({
        "Model": col,
        "Routes_Used": n_routes,
        "MAE": avg_mae,
        "MSE": avg_mse,
        "RMSE": avg_rmse
    })

metrics_df = pd.DataFrame(results)
metrics_df

,Model,Routes_Used,MAE,MSE,RMSE
0,Naive_Est,90,639.986121,1.400594e+07,2570.376643
1,EM_Est,90,401.511324,5.631749e+06,1639.316448
2,MARSS_Est,90,408.178710,5.006847e+06,1588.764017


In [10]:
pair_results = []

for (o, d), g in df_all.groupby(["Origin", "Destination"]):

    y_true = g["Final_True_Demand"]

    row = {
        "Origin": o,
        "Destination": d
    }

    for col in models:

        mask = g[col].notna()

        mse = np.mean((y_true[mask] - g[col][mask])**2)
        mae = np.mean(np.abs(y_true[mask] - g[col][mask]))

        row[f"{col}_MAE"] = mae
        row[f"{col}_MSE"] = mse

    pair_results.append(row)

pair_df = pd.DataFrame(pair_results)
pair_df.to_csv("pairwise_metrics.csv", index=False)
pair_df.head()

,Origin,Destination,Naive_Est_MAE,Naive_Est_MSE,EM_Est_MAE,EM_Est_MSE,MARSS_Est_MAE,MARSS_Est_MSE
0,CAN,CGO,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00
1,CAN,HKG,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00
2,CAN,JFK,488.711375,8.762156e+06,298.257253,3.674276e+06,296.802366,3.277657e+06
3,CAN,LAX,1303.304745,3.024709e+07,836.084025,1.306145e+07,822.112054,1.125157e+07
4,CAN,NRT,396.250608,5.539497e+06,253.601041,2.165571e+06,280.438133,2.377481e+06


In [11]:
pair_df["Best_Model"] = pair_df[
    ["Naive_Est_MAE", "EM_Est_MAE", "MARSS_Est_MAE"]
].idxmin(axis=1)

pair_df[["Origin", "Destination", "Best_Model"]]

,Origin,Destination,Best_Model
0,CAN,CGO,Naive_Est_MAE
1,CAN,HKG,Naive_Est_MAE
2,CAN,JFK,MARSS_Est_MAE
3,CAN,LAX,MARSS_Est_MAE
4,CAN,NRT,EM_Est_MAE
...,...,...,...
85,SIN,LAX,MARSS_Est_MAE
86,SIN,NRT,EM_Est_MAE
87,SIN,ORD,MARSS_Est_MAE
88,SIN,PEK,EM_Est_MAE
